In [4]:
from ultralytics import YOLO
import numpy as np
import cv2
import torch
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch import nn
import torch
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt
import timm
import torch.optim as optim

/home/hice1/axu39/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# CONFIG
MAX_SIZE = 512
yolo_path = "../yolo/yoloobb.pt"
test_image_id = 1075
training_data_dir = "/home/hice1/axu39/lizard/Lizard_Toepads/stacked-hourglass/data/hrnet"
test_data_dir = "/storage/ice-shared/cs8903onl/hourglass-data/raw_data/test_data"
imgdir = "/storage/ice-shared/cs8903onl/hourglass-data/raw_data/miami_fall_24_jpgs"
tps_data_dir = "/storage/ice-shared/cs8903onl/hourglass-data/raw_data/tps_files"

def get_img(img_id):
    return cv2.imread(f"{imgdir}/{img_id}.jpg")

def get_tps_coords(img_id, img):
    tps_classes = ["finger", "toe"]
    ret = {}
    h, w = img.shape[:2]
    for c in tps_classes:
        fp = f"{tps_data_dir}/{img_id}_{c}.TPS"
        coordinates = []
        skip = 2
        with open(fp, "r") as f:
            for line in f:
                line = line.strip()
                if not line or "=" in line:
                    continue
                parts = line.split()
                if len(parts) == 2:
                    if skip > 0:
                        skip -= 1
                        continue
                    try:
                        x , y = map(float, parts)
                        
                        coordinates.append((x, h - 1 - y))
                        #print((x, y))
                    except ValueError:
                        continue
        ret[c] = coordinates
    return ret

def tps_to_heatmap(tps, crop, show=False, sigma=10):
    h, w = crop.shape[:2]
    heatmap = []

    yy, xx = np.mgrid[0:h, 0:w]
    
    for i, (x, y) in enumerate(tps):
        hm = np.zeros((h,w), dtype=np.float32)
        x = float(x)
        y = float(y)
        g = np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * sigma**2))
        hm += g
        #hm /= hm.max() + 1e-8
        heatmap.append(hm);

        if show:
            copy = crop.copy()
            heatmap_uint8 = (hm * 255).astype(np.uint8)
            heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
            
            if heatmap_color.shape[:2] != copy.shape[:2]:
                heatmap_color = cv2.resize(heatmap_color, (copy.shape[1], copy.shape[0]))
            
            overlay = cv2.addWeighted(copy, 0.6, heatmap_color, 0.4, 0)
            
            cv2.imshow("Overlay", overlay)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
    return np.array(heatmap).transpose(1, 2, 0)

def process_images():
    dir_path = Path(imgdir)
    count = 0
    for file in dir_path.iterdir():
        print(f"Processing file {count}", end="\r", flush=True)
        try:
            if ".jpg" in file.name:
                imgid = file.name.replace(".jpg", "")
                if int(imgid) > 1000:
                    process_image(imgid, count < 10)
            count += 1
        except Exception as e:
            count += 1
            print()
            print(f"Failed to process file {file}: {e}")
            #continue
            break

def generate_overlay(crop, heatmaps):
    # Step 1: Combine heatmaps across channels (max intensity)
    combined = np.max(heatmaps, axis=2)  # shape (H,W), still float32 [0,1]

    # Step 2: Convert to uint8 0-255
    combined_uint8 = (np.clip(combined, 0, 1) * 255).astype(np.uint8)

    # Step 3: Apply color map
    heatmap_color = cv2.applyColorMap(combined_uint8, cv2.COLORMAP_JET)  # (H,W,3) uint8

    # Step 4: Resize if needed (match crop)
    if heatmap_color.shape[:2] != crop.shape[:2]:
        heatmap_color = cv2.resize(heatmap_color, (crop.shape[1], crop.shape[0]), interpolation=cv2.INTER_LINEAR)

    # Step 5: Blend overlay
    overlay = cv2.addWeighted(crop, 0.6, heatmap_color, 0.4, 0)

    return overlay

def crop_toe_boxes(r, image, g_coords, show=False, output_name="output"):
    target_classes = [2, 3] #["bot_finger", "bot_toe"]
    test_classes = [0, 1] #["up_finger", "up_toe"]
    classmap = {2: "finger", 3: "toe"}
    crops = []  # store cropped images
    coords_list = []  # store corresponding coordinates (for later heatmaps)
    tps = []
    result = r[0]  # first image
    boxes = result.boxes
    
    # Convert to numpy arrays for convenience
    xyxy = boxes.xyxy.cpu().numpy()   # shape (N, 4)
    cls_ids = boxes.cls.cpu().numpy() # shape (N,)
    conf = boxes.conf.cpu().numpy()   # optional if you want confidence filtering
    
    # Loop and filter
    for (x1, y1, x2, y2), cls_id in zip(xyxy, cls_ids):
        if int(cls_id) in target_classes:
            # Crop the image
            x1i, y1i, x2i, y2i = map(int, [x1, y1, x2, y2])
            crop = image[y1i:y2i, x1i:x2i].copy()  # copy to avoid referencing original image
            coords_list.append([x1i, y1i, x2i, y2i])  # store original coordinates
            
            l_coords = []
            valid = True
            for (x, y) in g_coords[classmap[int(cls_id)]]:
                x_local = x - x1
                y_local = y - y1
                # Check valid
                if not (0 <= x_local < (x2i - x1i) and 0 <= y_local < (y2i - y1i)):
                    valid = False
                    break

                l_coords.append((x_local, y_local))

            if not valid:
                continue
            
            crops.append(crop)
            tps.append(l_coords)

            if show:
                copy = crop.copy()
                for (lx, ly) in l_coords:
                    cv2.circle(copy, (int(round(lx)), int(round(ly))), 5, (0, 0, 255), -1)
                cv2.imshow(f"Crop Class {int(cls_id)}", copy)
                cv2.waitKey(0)   # waits for a key press
                cv2.destroyWindow(f"Crop Class {int(cls_id)}")
        elif int(cls_id) in test_classes:
            x1i, y1i, x2i, y2i = map(int, [x1, y1, x2, y2])
            crop = image[y1i:y2i, x1i:x2i].copy()
            outpath = f"{test_data_dir}/{output_name}_{classmap[int(cls_id)+2]}.jpg"
            #print(outpath)
            cv2.imwrite(outpath, crop)
            
    return crops, coords_list, tps

def process_image(imgid, save_crop=False):

    model = YOLO(yolo_path)

    img = get_img(imgid)
    tps = get_tps_coords(imgid, img)

    results = model(img, verbose=False)

    crops, box_coords, local_tps_coords = crop_toe_boxes(
        results, img, tps, output_name=imgid
    )

    for i, crop in enumerate(crops):

        if i > 1:
            break

        base_transform = A.Compose(
            [
                A.LongestMaxSize(max_size=MAX_SIZE),
                A.PadIfNeeded(
                    MAX_SIZE,
                    MAX_SIZE,
                    border_mode=cv2.BORDER_CONSTANT
                ),
            ],
            keypoint_params=A.KeypointParams(
                format="xy",
                remove_invisible=False
            ),
        )

        aug = base_transform(
            image=crop,
            keypoints=local_tps_coords[i]
        )

        img_aug = aug["image"]
        kps_aug = np.array(aug["keypoints"])

        # generate heatmaps
        heatmaps = tps_to_heatmap(kps_aug, img_aug, sigma=6)

        if heatmaps.shape[2] != 9:
            print()
            print(f"Mismatch found: {heatmaps.shape}")
            continue

        ipath = f"{training_data_dir}/crops/{imgid}_{i}_b.jpg"
        hpath = f"{training_data_dir}/heatmaps/{imgid}_{i}_b.pt"

        if save_crop:
            overlay = generate_overlay(img_aug, heatmaps)
            cv2.imwrite(ipath, overlay)

        torch.save(
            {
                "image": torch.from_numpy(img_aug).permute(2,0,1).to(torch.uint8),
                "heatmap": torch.from_numpy(heatmaps).permute(2,0,1).to(torch.float32),
                "tps": torch.from_numpy(kps_aug).to(torch.float32),
            },
            hpath,
        )

In [6]:
process_images()

Processing file 0
Failed to process file /storage/ice-shared/cs8903onl/hourglass-data/raw_data/miami_fall_24_jpgs/1001.jpg: PytorchStreamReader failed reading file data/307: invalid header or archive is corrupted
